Use cross-validation techniques for better model evaluation by implementing stratified k-fold cross-validation, Leave-One-Out Cross-Validation (LOOCV) for small datasets, and comparing performance stability across different folds.

In [6]:

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate, LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import joblib
import os


In [7]:
save_folder = "task31_saved_files"
preprocessor = joblib.load(os.path.join(save_folder, "preprocessor.pkl"))
feature_config = joblib.load(os.path.join(save_folder, "feature_config.pkl"))
x_test = joblib.load(os.path.join(save_folder, "x_test.pkl"))
y_test = joblib.load(os.path.join(save_folder, "y_test.pkl"))
x_train = joblib.load(os.path.join(save_folder, "x_train.pkl"))
y_train = joblib.load(os.path.join(save_folder, "y_train.pkl"))
print("All saved files loaded successfully!")

All saved files loaded successfully!


In [8]:
save_folder = "task31_saved_files"

x_train_processed = preprocessor.transform(x_train)
x_test_processed = preprocessor.transform(x_test)

print("Processed training shape:", x_train_processed.shape)
print("Processed testing shape:", x_test_processed.shape)

Processed training shape: (18786, 3049)
Processed testing shape: (4697, 3049)


In [9]:
print("Training features:", x_train.shape)
print("Training target:", y_train.shape)

print("\nClass distribution:")
print(y_train.value_counts())

print("\nPreprocessor:", type(preprocessor).__name__)

Training features: (18786, 15)
Training target: (18786,)

Class distribution:
recommend_flag
1    15448
0     3338
Name: count, dtype: int64

Preprocessor: ColumnTransformer


Create a leakage-safe pipeline

In [10]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression

In [11]:
cv_model = Pipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(penalty='l2', max_iter=2000, random_state=42))
])

Implement Stratified 5-Fold Cross-Validation

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [13]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(cv_model, x_train, y_train, cv=skf, scoring=scoring, return_train_score=False, n_jobs=-1, verbose=1)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    9.4s remaining:   14.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    9.5s finished


Compare performance across folds

In [14]:
fold_results = pd.DataFrame({
    "Fold": range(1,6),
    "Accuracy": cv_results['test_accuracy'],
    "Precision": cv_results['test_precision'],
    "Recall": cv_results['test_recall'],
    "F1 Score": cv_results['test_f1'],
    "ROC AUC": cv_results['test_roc_auc']
})

display(fold_results.round(4))

,Fold,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,1,0.9449,0.9881,0.9443,0.9657,0.9823
1,2,0.9375,0.9867,0.9365,0.9610,0.9782
2,3,0.9452,0.9881,0.9446,0.9659,0.9816
3,4,0.9388,0.9857,0.9392,0.9619,0.9827
4,5,0.9375,0.9884,0.9350,0.9609,0.9800


In [15]:
stability_summary = fold_results.drop(columns=["Fold"]).agg(["mean", "std"]).T

stability_summary.columns = ["Mean", "Standard Deviation"]

display(stability_summary.round(4))

,Mean,Standard Deviation
Accuracy,0.9408,0.0040
Precision,0.9874,0.0012
Recall,0.9399,0.0044
F1 Score,0.9631,0.0025
ROC AUC,0.9810,0.0019


Select a small sample for LOOCV

In [16]:
from sklearn.model_selection import train_test_split

In [17]:
x_loo, _, y_loo, _ = train_test_split(x_train, y_train, test_size=100, stratify=y_train, random_state=42)

print("LOO training features:", x_loo.shape)
print("\nClass distribution in LOO training set:")
print(y_loo.value_counts())

LOO training features: (18686, 15)

Class distribution in LOO training set:
recommend_flag
1    15366
0     3320
Name: count, dtype: int64


Implement LOOCV

In [ ]:
loo = LeaveOneOut()
loo_scores = cross_validate(cv_model, x_loo, y_loo, cv=loo, scoring='accuracy', return_train_score=False, n_jobs=-1, verbose=1)

print("Total LOOCV iterations:", len(loo_scores))
print("LOOCV Accuracy:", round(loo_scores.mean(), 4))

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   25.4s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:  3.9min
[Parallel(n_jobs=-1)]: Done 784 tasks      | elapsed:  6.8min
[Parallel(n_jobs=-1)]: Done 1234 tasks      | elapsed: 10.7min
[Parallel(n_jobs=-1)]: Done 1784 tasks      | elapsed: 16.2min
[Parallel(n_jobs=-1)]: Done 2434 tasks      | elapsed: 22.7min
[Parallel(n_jobs=-1)]: Done 3184 tasks      | elapsed: 30.4min
